<a href="https://www.kaggle.com/code/kaoutharhamdan/nlp-pipeline-week-1?scriptVersionId=318712638" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# NLP Pipeline - Practice

## Installer les prérequis

In [ ]:
!pip install requests beautifulsoup4 selenium webdriver-manager PyPDF2 spacy nltk
!python -m spacy download fr_core_news_sm

In [ ]:
!pip install PyPDF2


In [ ]:
!pip install -U spacy
!python -m spacy download fr_core_news_sm

## Exemple 1 : 

In [ ]:
import requests
from bs4 import BeautifulSoup
import spacy
import PyPDF2
from nltk.stem import SnowballStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

# Chargement du modèle linguistique français de spaCy
# Ce modèle contient les dictionnaires pour le POS tagging, la lemmatisation et le NER
nlp = spacy.load("fr_core_news_sm")

# -----------------------------------------------------------------------------
# ÉTAPE 1 : COLLECTE DES DONNÉES
# Rassembler le texte brut à partir de sources variées (Web, PDF, etc.)
# -----------------------------------------------------------------------------
def collect_data_web(url):
    """Récupère le texte brut depuis une page HTML."""
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    return " ".join([p.text for p in soup.find_all('p')])

# -----------------------------------------------------------------------------
# ÉTAPE 2 : PRÉTRAITEMENT DU TEXTE
# Prépare le texte : Nettoyage, Tokenisation, Normalisation et Réduction
# -----------------------------------------------------------------------------
def preprocess_text(raw_text):
    # A. NETTOYAGE & NORMALISATION : Passage en minuscule et retrait des espaces inutiles
    text = raw_text.lower().strip()
    
    # B. TOKENISATION & LEMMATISATION : spaCy découpe en tokens et trouve la forme canonique
    doc = nlp(text)
    
    # C. FILTRAGE : On retire les 'Stopwords' (mots vides) et la ponctuation
    # On ne garde que le 'lemme' (racine linguistique) pour réduire la complexité
    tokens_valides = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct]
    
    return doc, " ".join(tokens_valides)

# -----------------------------------------------------------------------------
# ÉTAPE 3 & 4 : ANALYSE SYNTAXIQUE & SÉMANTIQUE
# Comprendre la structure (POS: Part of Speech) et identifier les entités (NER)
#          --> Part-of-Speech (POS) tagging is the process of assigning grammatical categories 
#             (like noun, verb, adjective, etc.) to each word in a sentence based on its definition and context. 
# -----------------------------------------------------------------------------
def analyze_linguistics(doc):
    print(f"{'Mot':<12} | {'Nature (POS)':<10} | {'Dépendance'}")
    print("-" * 40)
    for token in list(doc)[:5]: # Exemple sur les 5 premiers mots
        # ÉTAPE 3 : POS Tagging (Nom, Verbe, Adjectif...)
        print(f"{token.text:<12} | {token.pos_:<10} | {token.dep_}")

    # ÉTAPE 4 : Reconnaissance d'Entités Nommées (NER)
    print("\n--- Entités Nommées détectées ---")
    for ent in doc.ents:
        print(f"Entité : {ent.text} ({ent.label_})")

# -----------------------------------------------------------------------------
# ÉTAPE 5 : REPRÉSENTATION DES DONNÉES (VECTORISATION)
# Convertir le texte en nombres pour que la machine puisse "calculer"
# -----------------------------------------------------------------------------
def vectorize_data(corpus):
    # Utilisation de TF-IDF pour donner du poids aux mots significatifs
    vectorizer = TfidfVectorizer()
    X = vectorizer.fit_transform(corpus)
    return X, vectorizer

# -----------------------------------------------------------------------------
# ÉTAPE 6 & 7 : MODÉLISATION ET ÉVALUATION
# Appliquer un algorithme et tester ses performances
# -----------------------------------------------------------------------------
'''
def train_and_evaluate(X, y):
    # Exemple de classification (Spam vs Non-Spam)
    model = MultinomialNB()
    model.fit(X, y)
    
    # Évaluation (Précision, Rappel, F1-score...)
    score = model.score(X, y)
    print(f"\nPrécision du modèle : {score * 100:.2f}%")
    return model
'''
# =============================================================================
# EXÉCUTION DU PIPELINE
# =============================================================================

# Texte d'exemple (Collecte)
raw_content = "Apple a été fondée par Steve Jobs en Californie. Il a créé l'iPhone."

# Prétraitement (Étape 2)
document_spacy, text_clean = preprocess_text(raw_content)

# Analyse (Étape 3 & 4)
analyze_linguistics(document_spacy)

# Représentation (Étape 5)
# (Ici sur un mini-corpus pour l'exemple)
X_vectors, feat_vec = vectorize_data([text_clean])


print("\nPipeline terminé avec succès.")


1. Tokenisation (Le découpage)
Le texte est segmenté en unités atomiques. Le modèle fr_core_news_sm est assez intelligent pour comprendre que "d'ia" doit être séparé en ["d'", "ia"].

2. Stopwords (Le nettoyage)
Le code utilise la propriété .is_stop de spaCy. Les mots comme "les", "de", "la" ou "à" sont évacués car ils sont statistiquement trop fréquents pour aider à classer le texte.

3. Lemmatisation (L'unification)
C'est ici que l'analyse linguistique prend tout son sens. Le mot "étudient" est ramené à "étudier" et "ingénieurs" devient "ingénieur".

Pourquoi ? Si vous avez un autre texte qui parle d'un "ingénieur", le modèle saura qu'il s'agit du même sujet, même si le pluriel diffère.

4. TF-IDF (La signature numérique)
Le tableau final montre le poids de chaque mot.

TF (Term Frequency) : Plus le mot "IA" est présent dans votre texte, plus son score monte.

IDF (Inverse Document Frequency) : Si le mot "FST" est présent dans tous les documents de votre présentation, son score baissera car il n'est plus discriminant (il ne permet plus de différencier un document d'un autre).

Résultat final : Votre phrase est passée d'une structure complexe et humaine à un vecteur de probabilités mathématiques prêt à être injecté dans un classifieur (comme une Régression Logistique ou Naive Bayes).

# Exemple 2: 

In [ ]:
import spacy
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# Chargement du modèle français
nlp = spacy.load("fr_core_news_sm")

def trace_nlp_pipeline(text):
    print(f"--- TEXTE BRUT ---\n'{text}'\n")
    
    # 1. NORMALISATION & TOKENISATION
    doc = nlp(text.lower().strip())
    tokens_bruts = [token.text for token in doc]
    print(f"1. Tokenisation (Minuscules) :\n{tokens_bruts}\n")
    
    # 2. FILTRAGE (Stopwords & Ponctuation)
    tokens_filtres = [token.text for token in doc if not token.is_stop and not token.is_punct]
    stopwords_elimines = [token.text for token in doc if token.is_stop]
    print(f"2. Après retrait des Stopwords :\n{tokens_filtres}")
    print(f"(Mots supprimés : {list(set(stopwords_elimines))})\n")
    
    # 3. LEMMATISATION
    # On transforme chaque mot filtré en sa forme dictionnaire (lemme)
    lemmes = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct]
    print(f"3. Après Lemmatisation (Racines) :\n{lemmes}\n")
    
    return " ".join(lemmes)

# --- EXEMPLE DÉTAILLÉ ---
phrase_test = "Les ingénieurs étudient les algorithmes d'IA à la FST de Tanger !"
texte_propre = trace_nlp_pipeline(phrase_test)

# --- 4. ZOOM SUR TF-IDF (Vectorisation) ---
# Imaginons un mini-corpus pour calculer les poids
corpus = [texte_propre, "étudiant fst tanger", "algorithme ia complexe"]
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus)

# Affichage des scores pour le premier document
print("4. Représentation Numérique (TF-IDF) pour le document 1 :")
df_tfidf = pd.DataFrame(tfidf_matrix.toarray(), columns=vectorizer.get_feature_names_out())
print(df_tfidf.iloc[0].sort_values(ascending=False))

## Interprétation du résultat

Ce résultat montre la transformation complète d'une phrase humaine en une **signature mathématique** que l'IA peut traiter. Voici l'explication détaillée de chaque étape de votre pipeline :



### 1. Tokenisation (Le découpage)
Le texte brut est segmenté en unités minimales appelées **tokens**. 
* **Observation :** Le modèle a intelligemment séparé `"d'ia"` en `"d'"` et `"ia"`. 
* **Objectif :** Isoler chaque élément syntaxique pour pouvoir les traiter individuellement. La mise en minuscules évite que l'ordinateur ne traite "Les" et "les" comme deux mots différents.


### 2. Retrait des Stopwords (Le filtrage)
On élimine les "mots vides" (déterminants, prépositions). 
* **Observation :** Les mots `['de', 'les', 'la', 'à', "d'"]` ont disparu. 
* **Objectif :** Réduire le "bruit". Ces mots sont statistiquement trop fréquents dans la langue française pour aider à distinguer le sujet de la phrase. On ne garde que les **mots porteurs de sens** (les mots pleins).


### 3. Lemmatisation (L'unification)
C'est l'étape de normalisation linguistique. On ramène chaque mot à sa forme canonique (celle du dictionnaire).
* **Transformations :** * `ingénieurs` (pluriel) $\rightarrow$ **ingénieur** (singulier).
    * `étudient` (verbe conjugué) $\rightarrow$ **étudier** (infinitif).
* **Objectif :** Regrouper les variantes d'un même mot. Si un autre texte utilisait "ingénieur" au singulier, le modèle comprendrait qu'il s'agit du même concept.


### 4. TF-IDF (La pondération mathématique)
C'est ici que l'on transforme les mots en nombres. Le score **TF-IDF** indique l'importance d'un mot dans votre document par rapport à un ensemble de documents (le corpus).

#### Pourquoi certains scores sont-ils plus hauts (0.48 vs 0.36) ?
Le score dépend de deux facteurs :
1.  **TF (Term Frequency) :** Combien de fois le mot apparaît dans votre phrase.
2.  **IDF (Inverse Document Frequency) :** À quel point le mot est **rare** dans les autres documents de votre exemple.

* **Poids fort (0.48) pour "étudier" et "ingénieur" :** Cela signifie que ces mots sont très spécifiques à votre document 1 et n'apparaissent probablement pas (ou très peu) dans les autres documents de comparaison que vous avez fournis au `TfidfVectorizer`. Ils sont donc jugés **très représentatifs** du contenu.
* **Poids moyen (0.36) pour "ia", "fst", "tanger" :** Ces mots sont importants, mais s'ils apparaissent aussi dans les autres documents du corpus (comme "étudiant fst tanger"), leur score baisse car ils deviennent moins "uniques" pour différencier le document 1 des autres.
* **Score 0.00 pour "complexe" et "étudiant" :** Ces mots n'existent tout simplement pas dans votre phrase de test, même s'ils font partie du vocabulaire global du corpus.



### En bref
Votre phrase n'est plus du texte pour la machine, c'est un **vecteur dans un espace multidimensionnel**. Plus le score est élevé, plus le mot est considéré comme un "mot-clé" crucial pour caractériser votre document.

# Exemple 3

In [ ]:
import spacy
from nltk.stem import SnowballStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

# Chargement des outils
nlp = spacy.load("fr_core_news_sm")
stemmer = SnowballStemmer("french")

text_brut = "Les algorithmes d'IA analysent les données massives pour les chercheurs à Tanger."

# --- STEP 1 : Tokenisation & Stopwords ---
doc = nlp(text_brut.lower())
tokens_clean = [t for t in doc if not t.is_stop and not t.is_punct]
print(f"1. Après Stopwords : {[t.text for t in tokens_clean]}")

# --- STEP 2 : Stemming ---
stems = [stemmer.stem(t.text) for t in tokens_clean]
print(f"2. Stemming (Racines) : {stems}")

# --- STEP 3 : Lemmatisation ---
lemmas = [t.lemma_ for t in tokens_clean]
print(f"3. Lemmatisation (Dictionnaire) : {lemmas}")

# --- STEP 4 : TF-IDF ---
# On crée un mini-corpus pour que le calcul TF-IDF ait du sens
corpus = [" ".join(lemmas), "ia tanger fst", "algorithme donnée analyse"]
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus)

# Affichage des poids pour notre phrase (Document 0)
print("\n4. Poids TF-IDF (Importance des mots) :")
df = pd.DataFrame(tfidf_matrix.toarray(), columns=vectorizer.get_feature_names_out())
print(df.iloc[0].sort_values(ascending=False))

# Devoir 

- Extraire les données depuis un pdf
- Utiliser des textes arabes au lieu de francais

**Arbic Processing**

In [ ]:
# ─── Install only what's missing on Kaggle ───
!pip install stanza --quiet

import re, nltk, stanza
from nltk.stem.isri import ISRIStemmer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

# Download Arabic model (runs once, ~500MB — be patient)
stanza.download('ar')
nlp_ar = stanza.Pipeline('ar', processors='tokenize,pos,ner')

In [ ]:
stanza.download('ar', '/kaggle/working/stanza_resources')
nlp_ar = stanza.Pipeline('ar', processors='tokenize,pos,ner',
                          dir='/kaggle/working/stanza_resources')

In [ ]:
# etape 1 pretraitement 
import re 
def clean_text(text):
    #let only arabe words
     text = re.sub(r'[^\u0600-\u06FF\s]', '', text)
    # get ride off useless spaces and replace some variants 
     text = re.sub(r'\s+', ' ', text).strip()
     text = re.sub(r'[إأآا]', 'ا', text)
     text = re.sub(r'ة', 'ه', text)
     text = re.sub(r'ى', 'ي', text)
     return text


In [ ]:
def prepare_text(text):
    arabic_stops = set(stopwords.words('arabic'))
    tokens = nltk.word_tokenize(text)
    filtered = [t for t in tokens
                if re.match(r'[\u0600-\u06FF]+', t) and t not in arabic_stops]
    return filtered

    

In [ ]:
def stem_tokens(tokens):
    stemmer = ISRIStemmer()
    return [stemmer.stem(t) for t in tokens]

In [ ]:
def analyze_linguistics(text):
    doc = nlp_ar(text)
    print(f"\n{'Token':<20} | {'POS':<10} | {'Dépendance'}")
    print("-" * 45)
    for sentence in doc.sentences:
        for word in sentence.words[:5]:
            print(f"{word.text:<20} | {word.upos:<10} | {word.deprel}")

    print("\n--- Entités Nommées ---")
    for sentence in doc.sentences:
        for ent in sentence.ents:
            print(f"  {ent.text}  →  {ent.type}")
            

In [ ]:
def vectorize(corpus):
    vectorizer = TfidfVectorizer()
    X = vectorizer.fit_transform(corpus)
    return X, vectorizer

In [ ]:
arabic_text = """في أعماق الإنسان صراع لا يهدأ، ليس بين الخير والشر كما يظن البسطاء، بل بين ما هو مألوف وما هو ممكن.

إنّ الإنسان لا يخشى الحقيقة، بل يخشى ما تتطلبه الحقيقة من تحوّل.

نحن لا نبحث عن المعنى، بل نهرب من الفراغ الذي يكشفه غياب المعنى.

من أراد أن يرى أبعد، عليه أن يتحمّل العزلة؛ فالجماعة تمنح الدفء، لكنها تسلب الرؤية.

ليست القوة في السيطرة على الآخرين، بل في القدرة على إعادة خلق الذات رغم الانهيار.

كل يقين مريح هو قيد خفي، وكل شك صادق هو بداية حرية فريدريك نيتشه."""

In [ ]:
normalized   = clean_text(arabic_text)
tokens       = prepare_text(normalized)
stems        = stem_tokens(tokens)

In [ ]:
print("Texte normalisé :", normalized)
print("Tokens filtrés   :", tokens)
print("Racines (stems)  :", stems)

In [ ]:
analyze_linguistics(arabic_text)

X, vec = vectorize([" ".join(stems)])